<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-01-setup-and-iam/lesson-1.1-setup/notebooks/GCP_Capstone_1.1_Setup.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.1 Setting Up Your GCP AI Project
**Netsetos GenAI Engineering — GCP Capstone**

This notebook verifies your GCP project setup and runs your first Gemini calls.


## Step 1: Install Dependencies


In [ ]:
!pip install -q google-genai==2.21.0   # the only package this notebook imports; 2.3 adds Firestore, pinned to the kit's 2.30.0


## Step 2: Authenticate (if running on Colab)


In [ ]:
# Only needed on Google Colab (not Cloud Shell or Workbench)
import sys
if 'google.colab' in sys.modules:
    from google.colab import auth
    auth.authenticate_user()


## Step 3: Set Your Project


In [ ]:
PROJECT_ID = "documind-ai-YOUR-ID"  # <-- CHANGE THIS
# Gemini 3.x on Vertex AI is served ONLY from the global endpoint. A regional
# value like "us-central1" returns 404 "model not found in the specified region".
# (Regional services — Cloud Run, BigQuery, embeddings — still use a region.)
LOCATION = "global"

!gcloud config set project {PROJECT_ID}
print(f"Project: {PROJECT_ID}")

## Step 4: Verify & Enable APIs

In [ ]:
# GCP APIs are OFF by default (a security feature). This VERIFIES first and only
# enables what's MISSING — so if everything is already on you skip the enable
# call (and the serviceusage.enable permission it needs). Idempotent, safe to re-run.
import subprocess
assert PROJECT_ID != "documind-ai-YOUR-ID", "set PROJECT_ID in the Setup cell first"
# The 32 services the DocuMind kit enables (deploy/commands/lesson-12.1.sh) - every lesson to 12.8
# runs on this list; 1.2's impersonation needs iamcredentials, the UI's sign-in needs iap.
CAPSTONE_APIS = [
    "run.googleapis.com",             # Cloud Run
    "compute.googleapis.com",         # Compute Engine - VPC, connectors
    "vpcaccess.googleapis.com",       # Serverless VPC Access
    "pubsub.googleapis.com",          # Pub/Sub - the ingest topic and DLQ
    "artifactregistry.googleapis.com", # Artifact Registry
    "secretmanager.googleapis.com",   # Secret Manager
    "firestore.googleapis.com",       # Firestore - the one collection, chunks
    "storage.googleapis.com",         # Cloud Storage - the uploads bucket
    "aiplatform.googleapis.com",      # Vertex AI - Gemini, embeddings, tuning
    "documentai.googleapis.com",      # Document AI - Layout Parser / OCR
    "speech.googleapis.com",          # Speech-to-Text
    "texttospeech.googleapis.com",    # Text-to-Speech
    "dlp.googleapis.com",             # Sensitive Data Protection - the PII scan
    "iap.googleapis.com",             # Identity-Aware Proxy - the UI's sign-in
    "iamcredentials.googleapis.com",  # IAM Credentials - identity tokens by impersonation (1.2)
    "cloudbuild.googleapis.com",      # Cloud Build
    "cloudtrace.googleapis.com",      # Cloud Trace
    "monitoring.googleapis.com",      # Cloud Monitoring
    "logging.googleapis.com",         # Cloud Logging
    "billingbudgets.googleapis.com",  # Billing Budgets - the alert
    "bigquery.googleapis.com",        # BigQuery
    "discoveryengine.googleapis.com", # Discovery Engine - the Ranking API, Vertex AI Search
    "dataplex.googleapis.com",        # Dataplex
    "sqladmin.googleapis.com",        # Cloud SQL Admin
    "eventarc.googleapis.com",        # Eventarc
    "workflows.googleapis.com",       # Cloud Workflows
    "cloudscheduler.googleapis.com",  # Cloud Scheduler
    "cloudfunctions.googleapis.com",  # Cloud Functions
    "modelarmor.googleapis.com",      # Model Armor - prompt screening
    "cloudbilling.googleapis.com",    # Cloud Billing
    "cloudresourcemanager.googleapis.com", # Resource Manager
    "serviceusage.googleapis.com",    # Service Usage
]

def enabled_apis():
    # config.name is the short service id (aiplatform.googleapis.com); `name` would
    # be the full resource path (projects/N/services/...). Normalize either way to
    # the last path segment so the membership test below matches CAPSTONE_APIS.
    out = subprocess.run(
        ["gcloud", "services", "list", "--enabled",
         "--format=value(config.name)", f"--project={PROJECT_ID}"],
        capture_output=True, text=True,
    ).stdout
    return {ln.split("/")[-1].strip() for ln in out.splitlines() if ln.strip()}

enabled = enabled_apis()
missing = [a for a in CAPSTONE_APIS if a not in enabled]

if not missing:
    print(f"✅ All {len(CAPSTONE_APIS)} capstone APIs already enabled on {PROJECT_ID} — good to go.")
else:
    print(f"Enabling {len(missing)} missing API(s) on {PROJECT_ID} (~30-60s): {', '.join(missing)}")
    # Service Usage takes at most 20 services per call (4.8, F1) - chunk the list
    for k in range(0, len(missing), 20):
        subprocess.run(["gcloud", "services", "enable", *missing[k:k + 20], f"--project={PROJECT_ID}"], check=False)
    enabled = enabled_apis()  # re-check after enabling

# Final status — the critical APIs must be ACTIVE before Step 6 works
for api in CAPSTONE_APIS:
    print(f"  {'✅' if api in enabled else '⏳ still propagating — wait ~60s and re-run'} {api.split('.')[0]}")

## Step 5: Initialize Gemini Client


In [ ]:
from google import genai

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)
print("✅ Gemini client initialized")


> **Two clients from Module 2 onward.** This client (`location="global"`) is for Gemini 3.x *generation* only — `generate_content`, `count_tokens`, streaming, chats. **Embeddings, tuning and evaluation are regional-only**, so they need a second client: `genai.Client(enterprise=True, project=PROJECT_ID, location="us-central1")` for the course (`asia-south1` for India production data). Module 2 onward uses both clients side by side.


## Step 6: First Gemini Call + Token Usage


In [ ]:
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Explain what Google Cloud Platform is in exactly 2 sentences.'
)

print('✅ Gemini says:')
print(response.text)

# Print token usage (thinking tokens are billed at the OUTPUT rate)
usage = response.usage_metadata
thinking = getattr(usage, "thoughts_token_count", 0) or 0
print(f'\n📊 Token Usage:')
print(f'  Input tokens:    {usage.prompt_token_count}')
print(f'  Output tokens:   {usage.candidates_token_count}')
print(f'  Thinking tokens: {thinking}')
print(f'  Total tokens:    {usage.total_token_count}')

# Calculate cost — Gemini 3.6 Flash INTRODUCTORY rate ($0.75/M in, $3.75/M out)
# through 2026-12-31; standard rate $1.50/M in, $7.50/M out from 2027-01-01.
PRICE_IN, PRICE_OUT = 0.75, 3.75
USD_TO_INR = 85
cost = (usage.prompt_token_count * PRICE_IN
        + (usage.candidates_token_count + thinking) * PRICE_OUT) / 1_000_000
print(f'  Cost: ${cost:.6f} (₹{cost*USD_TO_INR:.4f}) — intro rate $0.75/$3.75 per 1M to 2026-12-31, USD_INR=85')
print(f'\n  With $500: {500/cost:,.0f} calls possible')
print('\n🎉 Setup complete! Your GCP GenAI environment is ready.')


## Step 7: Free Token Counter


In [ ]:
# count_tokens is FREE — no inference, no cost
texts = [
    'Hello',
    'Hyderabad is the capital of Telangana' * 10,
    'The transformer architecture uses self-attention' * 100,
]

print('📈 Token Counts (FREE — no API cost):')
for t in texts:
    result = client.models.count_tokens(model='gemini-3.6-flash', contents=t)
    # Input priced at the STANDARD rate ($1.50/M from 2027-01-01); intro rate is $0.75/M through 2026-12-31
    est_cost = result.total_tokens * 1.50 / 1_000_000 * USD_TO_INR
    print(f'  {result.total_tokens:>6,} tokens | {len(t):>6} chars | ₹{est_cost:.4f} input | {t[:40]}...')


## Step 8: Compare 3 Models


In [ ]:
import time

prompt = 'Explain the difference between SQL and NoSQL databases. Give 2 examples of each.'
# (name, input $/M, output $/M). 3.6 Flash at its INTRODUCTORY rate to 2026-12-31 - the same rate
# Cell 6 used, so the two tables agree; from 2027-01-01 the Flash row is 1.50 / 7.50 (standard).
models = [
    ('gemini-3.1-flash-lite', 0.25, 1.50),
    ('gemini-3.6-flash', 0.75, 3.75),
    ('gemini-3.1-pro-preview', 2.00, 12.00),
]

print(f"{'Model':<30} {'Tokens':>7} {'Latency':>9} {'Rs':>9}")
print('-' * 60)
for name, ip, op in models:
    t0 = time.time()
    try:
        r = client.models.generate_content(model=name, contents=prompt)
        ms = (time.time()-t0)*1000
        u = r.usage_metadata
        cost = (u.prompt_token_count*ip + (u.candidates_token_count + (getattr(u, 'thoughts_token_count', 0) or 0))*op)/1e6*85
        print(f'{name:<30} {u.total_token_count:>7} {ms:>7.0f}ms Rs.{cost:>6.4f}')
    except Exception as e:
        print(f'{name:<30} ERROR: {e}')


## ✅ Lesson 1.1 Complete!

- ✅ GCP project created and configured
- ✅ Billing linked (console) - the budget alert is Step 2 on the lesson page
- ✅ 32 APIs enabled, twenty per call
- ✅ Gemini client initialized with `google-genai`
- ✅ First Gemini call with token usage and cost
- ✅ Free token counter tested
- ✅ 3 models compared (Flash-Lite, Flash, Pro)

**Next: Lesson 1.2 — IAM & Security for GenAI Projects**
